## Foundation Snippet- Build simple PMF and CDF classes
This snippet creates small custom versions of `Pmf` and `Cdf` using pandas Series. It helps students understand how frequency counts become probabilities, how probabilities accumulate into a CDF, and how a CDF can be converted back into a PMF. This foundation is useful across chapters because PMFs and CDFs appear repeatedly in distribution analysis, percentile work, sampling, and model comparison.

In [1]:

"""
Context: defines simple PMF and CDF classes for repeated use across chapters
Plain-English: builds probability distributions from sequences, converts PMF to CDF, and converts CDF back to PMF
Real-world: used to summarize data values and their probabilities in a clean table-like form
Reappears: yes in distribution, percentile, sampling, and model comparison work
"""

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


def underride(d, **kwargs):
    for key, val in kwargs.items():  
        # go through each default option

        d.setdefault(key, val)  
        # add default only if key is missing

    return d  
    # return updated dictionary


class Pmf(pd.Series):

    @property
    def _constructor(self):
        return Pmf  
        # keep pandas operations returning Pmf when possible

    @staticmethod
    def from_seq(
        seq,
        normalize=True,
        sort=True,
        ascending=True,
        dropna=True,
        na_position="last",
        **kwargs
    ):
        series = pd.Series(seq).value_counts(
            normalize=normalize,
            sort=False,
            dropna=dropna
        )  
        # count values, or compute probabilities if normalize=True

        pmf = Pmf(series, copy=False, **kwargs)  
        # convert counts or probabilities into Pmf

        if sort:
            pmf.sort_index(
                inplace=True,
                ascending=ascending,
                na_position=na_position
            )  
            # sort values in order

        return pmf  
        # return final PMF

    def normalize(self):
        total = self.sum()  
        # add all probabilities or counts

        self /= total  
        # divide each value by total

        return total  
        # return value used for normalization

    def make_cdf(self, **kwargs):
        normalize = kwargs.pop("normalize", False)  
        # decide whether final CDF should end at 1

        pmf = self.sort_index()  
        # sort values before cumulative sum

        cumulative = np.cumsum(pmf)  
        # add probabilities step by step

        cdf = Cdf(
            cumulative,
            index=pmf.index.copy(),
            **kwargs
        )  
        # create CDF using same values as index

        if normalize:
            cdf.normalize()  
            # make final cumulative probability equal 1

        return cdf  
        # return CDF


class Cdf(pd.Series):

    @property
    def _constructor(self):
        return Cdf  
        # keep pandas operations returning Cdf when possible

    @property
    def qs(self):
        return self.index.values  
        # quantities as NumPy array

    @property
    def ps(self):
        return self.values  
        # probabilities as NumPy array

    def copy(self, deep=True):
        return Cdf(self, copy=deep)  
        # return a CDF copy

    def normalize(self):
        total = self.ps[-1]  
        # final cumulative value

        self /= total  
        # divide all cumulative values by final value

        return total  
        # return value used for normalization

    @staticmethod
    def from_seq(seq, normalize=True, sort=True, **kwargs):
        pmf = Pmf.from_seq(seq, normalize=False, sort=sort, **kwargs)  
        # make counts first, not probabilities

        return pmf.make_cdf(normalize=normalize)  
        # convert counts to CDF, then normalize if needed

    def forward(self, qs, **kwargs):
        underride(
            kwargs,
            kind="previous",
            copy=False,
            assume_sorted=True,
            bounds_error=False,
            fill_value=(0, 1),
        )  
        # set lookup rules for quantity to probability

        interp = interp1d(self.qs, self.ps, **kwargs)  
        # create forward lookup function

        return interp(qs)  
        # return cumulative probability for qs

    def inverse(self, ps, **kwargs):
        underride(
            kwargs,
            kind="next",
            copy=False,
            assume_sorted=True,
            bounds_error=False,
            fill_value=(self.qs[0], np.nan),
        )  
        # set lookup rules for probability to quantity

        interp = interp1d(self.ps, self.qs, **kwargs)  
        # create inverse lookup function

        return interp(ps)  
        # return quantity for ps

    __call__ = forward  
    # calling a CDF uses forward lookup

    quantile = inverse  
    # quantile uses inverse lookup

    def median(self):
        m = self.inverse(0.5)  
        # find value at 50% cumulative probability

        return m  
        # return median

    def iqr(self):
        low, high = self.inverse([0.25, 0.75])  
        # find first and third quartiles

        return high - low  
        # return interquartile range

    def make_pmf(self, **kwargs):
        normalize = kwargs.pop("normalize", False)  
        # decide whether PMF should sum to 1

        diff = np.diff(self, prepend=0)  
        # undo cumulative sum by subtraction

        pmf = Pmf(diff, index=self.index.copy(), **kwargs)  
        # create PMF using same values as index

        if normalize:
            pmf.normalize()  
            # make probabilities add to 1

        return pmf  
        # return PMF

    def step(self, **kwargs):
        underride(kwargs, drawstyle="steps-post")  
        # use step style for CDF plot

        self.plot(**kwargs)  
        # plot CDF


seq = [1, 2, 2, 3]  
# small example sequence

pmf = Pmf.from_seq(seq)  
# make PMF directly from sequence

cdf1 = Cdf.from_seq(seq)  
# make CDF directly from sequence

cdf2 = pmf.make_cdf(normalize=True)  
# make CDF from PMF

print(
    "PMF =\n", pmf
     )

print(
    "CDF from sequence =\n", cdf1
     )

print(
    "CDF from PMF =\n", cdf2
     )


"""
Interpretation:
Pmf.from_seq counts values and converts them into probabilities
Cdf.from_seq makes a PMF from counts, then converts it into a normalized CDF
pmf.make_cdf creates the same CDF from an existing PMF
qs means the data values stored in the index
ps means the probabilities stored in the values
forward lookup goes from value to cumulative probability
inverse lookup goes from cumulative probability to value
median uses inverse lookup at 0.5
iqr uses inverse lookup at 0.25 and 0.75
make_pmf converts a CDF back into probability masses
"""

PMF =
 1    0.25
2    0.50
3    0.25
Name: proportion, dtype: float64
CDF from sequence =
 1    0.25
2    0.75
3    1.00
Name: count, dtype: float64
CDF from PMF =
 1    0.25
2    0.75
3    1.00
Name: proportion, dtype: float64


'\nInterpretation:\nPmf.from_seq counts values and converts them into probabilities\nCdf.from_seq makes a PMF from counts, then converts it into a normalized CDF\npmf.make_cdf creates the same CDF from an existing PMF\nqs means the data values stored in the index\nps means the probabilities stored in the values\nforward lookup goes from value to cumulative probability\ninverse lookup goes from cumulative probability to value\nmedian uses inverse lookup at 0.5\niqr uses inverse lookup at 0.25 and 0.75\nmake_pmf converts a CDF back into probability masses\n'